Cost of a candidate against K: wall clock and peak memory on one T4.

**GPU T4 x2**, Internet on, attach `prepare-data-for-word-reranker`. Around 25 minutes.

In [ ]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K

COMMIT = K.sync(REPO_URL, BRANCH)
K.gpu_info()
env = K.prepare(COMMIT)

The encoder runs once per utterance whatever K is, so it is timed apart from decoding.

In [ ]:
rc = K.run(env, "bench.py",
           "--base_model", env.base_model,
           "--adapter", env.adapter,
           "--librispeech", env.librispeech / "test-clean",
           "--ks", "1,3,5,10,15,20,30",
           "--n_utts", 200,
           "--json", env.results / f"bench-{COMMIT}.json")
assert rc == 0, f"bench exit code {rc}, see the output above"